# 08. 가설검정

**과제 필수 항목.** 중요한 것은 "검정을 했다"가 아니라
**"이 상황에 왜 이 검정인가"를 설명할 수 있는 것**이다.

### 네 개의 가설

| # | 질문 | H0 | H1 | 검정 | 왜 이 검정인가 |
|---|---|---|---|---|---|
| **1** | 전이학습이 밑바닥 CNN보다 나은가 | 두 모델의 오분류율이 같다 | 다르다 | **McNemar** | **같은 테스트셋**을 두 모델이 봤다 → 짝지은 자료 |
| **2** | 전처리가 효과가 있는가 | 두 조건의 평균 macro-F1 이 같다 | 다르다 | **대응표본 t + 윌콕슨** | 같은 시드끼리 짝지은 5쌍 |
| **3** | 이 테스트 점수를 믿어도 되는가 | 증강 TEST 정확도 = 원본 정확도 | 다르다 | **두 비율 z검정** | 서로 **다른 표본** → 독립 |
| **4** | CAM 이 실제로 세포를 보는가 | CAM 질량비 = 면적비 | 질량비가 더 크다 | **대응표본 t (단측)** | 같은 이미지에서 잰 두 값 |

유의수준 α = 0.05. 검정을 4번 하므로 마지막에 **Holm 보정**을 적용한다.

### 검정을 고르는 기준 (이걸 설명할 수 있어야 한다)

```
비교하려는 두 값이 같은 대상에서 나왔나?
 ├─ 예 (짝지은 자료) ─┬─ 이진 결과(맞음/틀림) → McNemar
 │                    └─ 연속값(점수)         → 대응표본 t검정 / 윌콕슨
 └─ 아니오 (독립 표본) ┬─ 비율 비교           → 두 비율 z검정 / 카이제곱
                       └─ 평균 비교           → 독립표본 t검정 / 맨-휘트니
```

**같은 테스트셋을 쓴 두 모델을 독립표본 검정으로 비교하는 것이 가장 흔한 실수다.**
같은 이미지들이므로 두 모델의 결과는 강하게 연관돼 있고, 그 연관을 무시하면 검정력을 잃는다.

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from scipy import stats

import wbc
wbc.use_korean_font()
wbc.NUM_WORKERS = 0

cfg = wbc.load_cfg()
ALPHA = 0.05
print('최종 모델:', cfg['model_name'], '/', cfg['preset'], '/', cfg['image_size'], 'px')

## 검정 1 — 전이학습 vs 밑바닥 CNN (McNemar)

> **H0**: 두 모델의 오분류율이 같다
> **H1**: 다르다 (양측)

### McNemar 가 하는 일

두 모델의 결과를 2×2 표로 정리한다.

| | B 맞음 | B 틀림 |
|---|---|---|
| **A 맞음** | n₁₁ | **b** |
| **A 틀림** | **c** | n₀₀ |

둘 다 맞히거나 둘 다 틀린 칸(n₁₁, n₀₀)은 **두 모델을 구분하는 정보가 없다.**
차이는 오직 **b 와 c** 에 있다. H0 가 참이면 b 와 c 는 대칭이어야 하므로

$$b \sim \text{Binom}(b+c,\ 0.5)$$

표를 직접 만들어 확인한다.

In [ ]:
model_final, m_final = wbc.eval_on_test('FINAL')        # 06 의 최종 모델
model_base,  m_base  = wbc.eval_on_test('A_밑바닥CNN')   # 02 의 베이스라인 A
print(wbc.summarize(m_final, '최종 전이학습 모델'))
print(wbc.summarize(m_base,  '밑바닥 CNN'))

y = m_final['trues']
A_ok = (m_final['preds'] == y)     # 모델 A = 전이학습
B_ok = (m_base['preds']  == y)     # 모델 B = 밑바닥

n11 = int(( A_ok &  B_ok).sum()); b = int(( A_ok & ~B_ok).sum())
c   = int((~A_ok &  B_ok).sum()); n00 = int((~A_ok & ~B_ok).sum())
display(pd.DataFrame([[n11, b], [c, n00]],
                     index=['A(전이학습) 맞음', 'A 틀림'],
                     columns=['B(밑바닥) 맞음', 'B 틀림']))
print(f'불일치 칸: b = {b} (A만 맞힘), c = {c} (B만 맞힘), 합 {b+c}')

In [ ]:
r1 = wbc.mcnemar_test(y, m_final['preds'], m_base['preds'])
print('=== 검정 1: McNemar ===')
print('  H0: 두 모델의 오분류율이 같다   H1: 다르다 (양측)')
print(f"  방법 {r1['method']}   통계량 {r1['statistic']:.4f}   p = {r1['p_value']:.4e}")
print(f"  정확도 {r1['acc_A']:.4f} (전이학습) vs {r1['acc_B']:.4f} (밑바닥)  차이 {r1['acc_diff']:+.4f}")
print('  판정 :', 'H0 기각 → 두 모델의 성능은 통계적으로 다르다'
      if r1['p_value'] < ALPHA else 'H0 기각 못함 → 차이를 입증하지 못했다')

r1b = wbc.permutation_test_acc(y, m_final['preds'], m_base['preds'], n_perm=5000)
print(f"  [검산] 순열검정 p = {r1b['p_value']:.4f}  (McNemar 와 비슷하면 결론이 안정적)")

**추가로 볼 것**: 04 에서 "최고 성능 모델"과 "선정 모델"이 달랐다면,
그 둘도 McNemar 로 비교해 **차이가 유의하지 않음**을 보이면
"더 싼 모델로 같은 성능을 냈다"는 강한 결론이 된다.

In [ ]:
# (선택) 04 의 최고 성능 모델과 최종 모델 비교
t = wbc.runs_table()
mrows = t[t.run_id.str.startswith('M_')].sort_values('val_macro_f1', ascending=False)
top_model = mrows.iloc[0].run_id if len(mrows) else None
if top_model and top_model[2:] != cfg['model_name']:
    _, m_top = wbc.eval_on_test(top_model)
    r_extra = wbc.mcnemar_test(y, m_final['preds'], m_top['preds'])
    print(f"최종({cfg['model_name']}) vs 최고성능({top_model[2:]})")
    print(f"  정확도 {r_extra['acc_A']:.4f} vs {r_extra['acc_B']:.4f}, p = {r_extra['p_value']:.4f}")
    print('  →', '차이 없음 → 싼 모델을 쓴 것이 정당하다' if r_extra['p_value'] >= ALPHA
          else '차이 있음 → 성능이 더 중요하면 최고 모델을 재검토한다')
else:
    print('최고 성능 모델이 곧 최종 모델이라 추가 비교 생략')

## 검정 2 — 전처리 효과 (대응표본 t검정 + 윌콕슨)

> **H0**: 최적 전처리와 무증강의 평균 검증 macro-F1 이 같다
> **H1**: 다르다 (양측)

03-4 에서 모은 **시드 5개짜리 점수쌍**을 쓴다.
같은 시드끼리 짝지어야 하는 이유: 시드가 초기화와 데이터 순서를 결정하므로,
같은 시드끼리 비교해야 **전처리 외의 변동을 상쇄**할 수 있다.

**p값만 보면 안 된다.**
- 표본이 5쌍뿐이라 검정력이 낮다 → 실제 차이가 있어도 못 잡을 수 있다
- 반대로 차이가 0.001 이어도 분산이 작으면 유의하게 나온다
→ **효과크기(Cohen's d)와 평균 차이의 실제 크기, 신뢰구간**을 함께 본다

In [ ]:
piv = pd.read_csv('seed_scores.csv', index_col=0, encoding='utf-8-sig')
display(piv.round(4))
a_col = [c for c in piv.columns if c.startswith('A_')][0]
b_col = [c for c in piv.columns if c.startswith('B_')][0]

# 대응표본이므로 '차이' 자체를 본다
d = piv[a_col].values - piv[b_col].values
print('시드별 차이 (A - B):', np.round(d, 4))
print(f'차이의 평균 {d.mean():+.4f}, 표준편차 {d.std(ddof=1):.4f}')

In [ ]:
r2 = wbc.paired_test(piv[a_col].values, piv[b_col].values, a_col, b_col)
print('=== 검정 2: 대응표본 검정 (최적 전처리 vs 증강 없음) ===')
print('  H0: 두 조건의 평균 macro-F1 이 같다   H1: 다르다 (양측)')
print(f"  n = {r2['n']}쌍   평균 {r2['mean_a']:.4f} vs {r2['mean_b']:.4f}   차이 {r2['mean_diff']:+.4f}")
print(f"  차이의 95% 신뢰구간 [{r2['ci95'][0]:+.4f}, {r2['ci95'][1]:+.4f}]")
print(f"  대응표본 t검정 : t = {r2['t']:.3f}, p = {r2['p_ttest']:.4f}")
print(f"  윌콕슨 부호순위 : W = {r2['w']:.1f}, p = {r2['p_wilcoxon']:.4f}   (정규성 가정이 약한 대안)")
print(f"  효과크기 Cohen's d = {r2['cohens_d']:.2f}  "
      f"({'큼' if abs(r2['cohens_d'])>0.8 else '중간' if abs(r2['cohens_d'])>0.5 else '작음'})")
print('  판정 :', 'H0 기각 → 전처리가 성능에 영향을 준다'
      if r2['p_ttest'] < ALPHA else 'H0 기각 못함 → 이 자료로는 차이를 입증하지 못했다')

> **"기각 못 함"은 "같다"가 아니다.** 귀무가설은 채택하는 것이 아니라 기각하지 못하는 것이다.
> 특히 n=5 는 검정력이 낮아, 실제 차이가 있어도 못 잡을 수 있다.
> → **"차이가 없다"가 아니라 "이 자료로는 차이를 입증하지 못했다"** 로 쓴다.
> 결론을 더 확실히 하려면 시드를 10~20개로 늘린다.

## 검정 3 — 이 테스트 점수를 믿어도 되는가 (두 비율 z검정) ⭐

**이번 프로젝트에서 가장 중요한 검정.**

01-5 에서 확인했듯 이 데이터셋은 원본 366장을 증강해 만든 것이라,
같은 원본에서 나온 사진이 TRAIN 과 TEST 양쪽에 있을 수 있다.
그렇다면 TEST 정확도는 실제 일반화 성능보다 부풀려진다.

확인 방법: **증강되지 않은 원본 이미지**(`external/`, 01-6 에서 생성)로 한 번 더 재고 비교한다.

> **H0**: 증강 TEST 정확도와 원본 이미지 정확도가 같다 (누수로 인한 과대추정이 없다)
> **H1**: 다르다 (양측)

두 평가셋은 **서로 다른 표본**이므로 McNemar 가 아니라 **두 비율 검정**을 쓴다.

$$z=\frac{\hat p_1-\hat p_2}{\sqrt{\hat p(1-\hat p)(1/n_1+1/n_2)}},\qquad
\hat p=\frac{k_1+k_2}{n_1+n_2}$$

In [ ]:
ext_loader, ext_ds = wbc.make_eval_loader('./external', image_size=int(cfg['image_size']),
                                          batch_size=64, center_crop_ratio=0.7)
print('외부 검증셋:', len(ext_ds), '장',
      dict(zip(ext_ds.classes, np.bincount(ext_ds.targets, minlength=4).tolist())))
m_ext = wbc.evaluate(model_final, ext_loader, nn.CrossEntropyLoss())
print(wbc.summarize(m_ext, '원본(외부) 평가'))
wbc.plot_confusion(m_ext['trues'], m_ext['preds'], normalize=True,
                   title='외부 검증셋(원본) 혼동행렬'); plt.show()

In [ ]:
k1 = int((m_final['preds'] == m_final['trues']).sum()); n1 = len(m_final['trues'])
k2 = int((m_ext['preds']   == m_ext['trues']).sum());   n2 = len(m_ext['trues'])
r3 = wbc.two_proportion_test(k1, n1, k2, n2)

print('=== 검정 3: 두 비율 z검정 ===')
print('  H0: 두 평가셋의 정확도가 같다   H1: 다르다 (양측)')
print(f'  증강 TEST : {k1}/{n1} = {r3["p1"]:.4f}')
print(f'  원본      : {k2}/{n2} = {r3["p2"]:.4f}')
print(f'  차이 {r3["diff"]:+.4f}   95% 신뢰구간 [{r3["ci95"][0]:+.4f}, {r3["ci95"][1]:+.4f}]')
print(f'  z = {r3["z"]:.3f}   p = {r3["p_value"]:.4e}')
print('  판정 :', 'H0 기각 → 두 값이 통계적으로 다르다'
      if r3['p_value'] < ALPHA else 'H0 기각 못함 → 차이를 입증하지 못했다')

ci_t = wbc.bootstrap_ci(m_final['trues'], m_final['preds'], 'accuracy')
ci_e = wbc.bootstrap_ci(m_ext['trues'],   m_ext['preds'],   'accuracy')
plt.figure(figsize=(5.5, 3.2))
plt.errorbar([0, 1], [ci_t['point'], ci_e['point']],
             yerr=[[ci_t['point']-ci_t['lo'], ci_e['point']-ci_e['lo']],
                   [ci_t['hi']-ci_t['point'], ci_e['hi']-ci_e['point']]],
             fmt='o', capsize=6, ms=8)
plt.xticks([0, 1], ['증강 TEST', '원본(외부)']); plt.ylabel('정확도')
plt.title('두 평가셋의 정확도와 95% 신뢰구간'); plt.grid(alpha=.3); plt.show()

### 해석 지침 — 결과에 따라 이렇게 쓴다

**H0 기각 + 원본 정확도가 낮다면**
→ 증강 TEST 점수는 실제 일반화 성능을 **과대평가**한 것이다.
보고서에는 두 숫자를 **모두** 적고, "일반화 성능의 보수적 추정치는 원본 쪽"이라고 쓴다.

**H0 기각 못했다면**
→ "누수가 없다"가 아니라 **"과대추정의 증거를 찾지 못했다"** 로 쓴다.

**어느 쪽이든 반드시 함께 적을 것 — 혼입요인(confounder)**
1. 원본은 640×480 전체 시야라 세포가 여러 개일 수 있어 **문제 자체가 더 어렵다.**
   따라서 차이 전부를 누수 탓으로 돌릴 수 없다.
2. 원본 표본이 340장 내외로 작아 **검정력이 제한**된다.
3. 화각을 중앙 70% 자르기로 맞췄지만 완전히 같지는 않다.

> 이렇게 쓰면 "정확도 99%면 너무 높은 거 아닌가요?" 라는 질문에
> **"저희도 그렇게 봐서 검사했고, 여기까지가 말할 수 있는 범위입니다"** 라고 답할 수 있다.

## 검정 4 — CAM 이 실제로 세포를 보는가 (단측 대응표본 t검정)

> **H0**: E[r − a] = 0 (CAM 은 핵 영역을 특별히 보고 있지 않다)
> **H1**: E[r − a] > 0 (핵 영역에 집중한다) — **단측**

같은 이미지에서 두 값(CAM 질량비 r, 면적비 a)을 재므로 **대응표본**이다.
"더 큰가"만 궁금하므로 단측검정을 쓴다.

In [ ]:
cr = pd.read_csv('cam_ratios.csv', encoding='utf-8-sig')
ratio, area = cr['ratio'].values, cr['area'].values
t_stat, p_one = stats.ttest_rel(ratio, area, alternative='greater')
d = ratio - area
res_cam = dict(n=len(d), mean_diff=float(d.mean()), t=float(t_stat), p=float(p_one),
               cohens_d=float(d.mean() / d.std(ddof=1)))

print('=== 검정 4: 대응표본 t검정 (단측) ===')
print('  H0: CAM 질량비 = 면적비   H1: CAM 질량비가 더 크다')
print(f"  n = {res_cam['n']}장   r 평균 {ratio.mean():.4f}   a 평균 {area.mean():.4f}")
print(f"  평균 차이 {res_cam['mean_diff']:+.4f}   t = {res_cam['t']:.3f}   p = {res_cam['p']:.4e}")
print(f"  효과크기 Cohen's d = {res_cam['cohens_d']:.2f}")
print('  판정 :', 'H0 기각 → CAM 은 핵 영역에 유의하게 집중한다'
      if res_cam['p'] < ALPHA else 'H0 기각 못함 → 근거가 세포에 있다고 말할 수 없다')

plt.figure(figsize=(5.5, 3.4))
plt.hist(d, bins=30, color='tab:purple', alpha=.7)
plt.axvline(0, c='r', ls='--'); plt.xlabel('r - a (CAM 질량비 - 면적비)')
plt.ylabel('이미지 수'); plt.title('0보다 오른쪽에 치우쳐야 세포를 보는 것'); plt.grid(alpha=.3)
plt.show()

## 다중비교 보정 (Holm-Bonferroni)

검정을 4번 하면, 실제로는 아무 차이가 없어도 그중 하나가 우연히 p<0.05 가 될 확률이

$$1-0.95^4\approx 18.5\%$$

로 올라간다. **Holm-Bonferroni** 는 p값을 작은 것부터 정렬해
$k$번째에 $(m-k+1)$ 을 곱하는 방식으로 이를 보정한다.
Bonferroni 보다 덜 보수적이면서 오류율을 같은 수준으로 통제한다.

In [ ]:
tests = [('1. 전이학습 vs 밑바닥CNN (McNemar)', r1['p_value']),
         ('2. 전처리 효과 (대응표본 t)',        r2['p_ttest']),
         ('3. 증강TEST vs 원본 (두 비율)',      r3['p_value']),
         ('4. CAM 이 핵을 보는가 (단측 t)',     res_cam['p'])]
hol = pd.DataFrame(wbc.holm_correction([p for _, p in tests], [n for n, _ in tests], alpha=ALPHA))
hol.columns = ['가설', 'p (원본)', 'p (Holm 보정)', '기각']
hol['판정'] = np.where(hol['기각'], '유의 (H0 기각)', '유의하지 않음')
display(hol[['가설', 'p (원본)', 'p (Holm 보정)', '판정']]
        .style.format({'p (원본)': '{:.3e}', 'p (Holm 보정)': '{:.3e}'}))

## 최종 정리

In [ ]:
summary = pd.DataFrame([
    dict(항목='무작위 기준선',              정확도=0.25,                macroF1=np.nan),
    dict(항목='밑바닥 CNN (베이스라인)',     정확도=m_base['accuracy'],  macroF1=m_base['macro_f1']),
    dict(항목=f"최종 모델 ({cfg['model_name']}) — 증강 TEST",
         정확도=m_final['accuracy'], macroF1=m_final['macro_f1']),
    dict(항목='최종 모델 — 원본(외부) 검증',  정확도=m_ext['accuracy'],   macroF1=m_ext['macro_f1']),
]).round(4)
display(summary)

t = wbc.runs_table()
print(f"\n최종 모델 : {cfg['model_name']} / 전처리 {cfg['preset']} / {cfg['image_size']}px / lr {cfg['lr']:g}")
print(f"TEST 정확도 {ci_t['point']:.4f}  95% CI [{ci_t['lo']:.4f}, {ci_t['hi']:.4f}]")
print(f"원본 정확도 {ci_e['point']:.4f}  95% CI [{ci_e['lo']:.4f}, {ci_e['hi']:.4f}]")
print(f"에폭당 시간 {float(t[t.run_id=='FINAL'].epoch_sec.iloc[0]):.0f}초 / 예산 180초")
print(f"누적 실험 수 {len(t)}건")
t.to_csv('results/전체실험기록.csv', index=False, encoding='utf-8-sig')
print('results/전체실험기록.csv 저장 (발표 부록용)')

### 보고서에 이렇게 쓴다

1. **문제**: 말초혈액 도말 이미지에서 백혈구 4종 분류. BASOPHIL 은 원본 3장뿐이라 제외.
2. **데이터**: TRAIN 9,957 / TEST 2,487, 클래스 균형, 320×240.
   **원본 366장을 증강해 만든 데이터**라는 점이 평가 설계를 좌우한다.
3. **모델 선정**: "1 에폭 3분" 예산을 속도 프로브로 먼저 확인해 후보를 거르고,
   그중 성능이 최고와 통계적으로 구분되지 않으면서 가장 빠른 모델을 선택. (04)
4. **전처리**: 데이터에 회전 증강이 이미 적용되어 있고 색이 클래스 신호이므로
   반전 위주 + 약한 색 변화. 프리셋 7종 비교로 근거 제시. (03)
5. **하이퍼파라미터**: 학습률·해상도·정규화·배치를 하나씩 바꿔 검증셋으로만 결정. (05)
6. **최종 성능**: TEST 정확도 X (95% CI [a,b]), macro-F1 Y.
   무작위 0.25, 밑바닥 CNN Z 대비. (06)
7. **CAM**: 판단 근거가 배경 적혈구가 아니라 백혈구 핵에 있음을 시각적·정량적으로 확인. (07)
8. **가설검정**: 자료 구조에 맞는 검정 4종 + Holm 보정. (08)
9. **한계**: ① 원본 단위 분리 미보장(검정 3으로 점검) ② BASOPHIL 미포함
   ③ 외부 검증셋이 작아 검정력 제한 ④ 단일 장비·단일 염색 조건.

**"이게 베스트다"라고 말할 때 반드시 함께 말할 것**
어떤 후보들과 비교했는지 · 어떤 기준으로 골랐는지 · 그 차이가 통계적으로 유의한지 ·
그리고 이 결론이 무너질 수 있는 조건은 무엇인지.